In [3]:
from instancepipeline import get_data_loaders

Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647


In [ ]:
train_loader,val_loader,test_loader = get_data_loaders()


In [6]:
import torch
import torch.nn

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
TARGET_MEAN = 2.28802128052746


In [9]:
import pandas as pd

In [11]:
all_targets = []

for waveforms, magnitudes in train_loader:

    magnitudes = magnitudes.to(device)

    magnitudes = magnitudes + TARGET_MEAN

    all_targets.append(
        magnitudes.cpu()
    )


all_targets = torch.cat(
    all_targets
).numpy()


targets = pd.DataFrame({
    "magnitude": all_targets
})


targets.head(5)

,magnitude
0,3.3
1,2.0
2,2.1
3,2.5
4,4.0


In [26]:
bin = 0

bins = []
counts = []

while bin < 6.6:

    bins.append(bin)

    count = (
        (targets["magnitude"] >= bin)
        & (targets["magnitude"] < bin + 0.1)
    ).sum()

    counts.append(count)

    bin = round(bin + 0.1, 1)

prior = pd.DataFrame({
    "bins": bins,
    "counts": counts
})

prior["probability"] = (
    prior["counts"] / prior["counts"].sum()
)

In [27]:
prior

,bins,counts,probability
0,0.0,95,0.000097
1,0.1,0,0.000000
2,0.2,695,0.000710
3,0.3,1363,0.001392
4,0.4,0,0.000000
...,...,...,...
61,6.1,50,0.000051
62,6.2,0,0.000000
63,6.3,0,0.000000
64,6.4,0,0.000000


In [20]:
prior["probability"] = (
    prior["counts"] / prior["counts"].sum()
)

In [21]:
prior

,bins,counts,probability
0,0.0,95,0.000097
1,0.1,0,0.000000
2,0.2,695,0.000710
3,0.3,1363,0.001392
4,0.4,0,0.000000
5,0.5,2076,0.002120
6,0.6,2804,0.002863
7,0.7,12394,0.012656
8,0.8,0,0.000000
9,0.9,10734,0.010961


In [23]:
print(targets["magnitude"].min())
print(targets["magnitude"].max())
print(prior["counts"].sum())
print(len(targets))

0.0
6.5
979323
979487


In [28]:
prior.to_csv("initialprior.csv",index=False)

In [30]:
from instancepipelineprior import get_data_loaders
train_loader, val_loader, test_loader = get_data_loaders()

_, y = next(iter(train_loader))

print("min:", y.min().item())
print("max:", y.max().item())
print("mean:", y.mean().item())

min: 0.6000000238418579
max: 3.9000000953674316
mean: 2.2835936546325684


In [31]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1



BIN_CENTERS = torch.arange(
    MIN_MAGNITUDE + BIN_WIDTH / 2,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    dtype=torch.float32
)
NUM_BINS = len(BIN_CENTERS)

In [33]:
from discretizedtrain import EarthquakeCNN, logits_to_magnitude

In [36]:
model = EarthquakeCNN()
model = model.to(device)

In [37]:
print(NUM_BINS)
print(BIN_CENTERS.shape)

data, target = next(iter(train_loader))
data = data.to(device)

with torch.no_grad():
    logits = model(data)
    predictions = logits_to_magnitude(logits, BIN_CENTERS.to(device))

print(logits.shape)
print(predictions.shape)
print(target.shape)
print(predictions[:5])
print(target[:5])

66
torch.Size([66])


torch.Size([128, 66])
torch.Size([128])
torch.Size([128])
tensor([3.2844, 3.2890, 3.2933, 3.2895, 3.2885], device='cuda:0')
tensor([2.0000, 1.7000, 2.3000, 1.9000, 2.5000])
